# 2. Model simulations

Simulates the goal model (Equation 3, parameters fitted to Fajen & Warren, 2003, held fixed) combined with
either collision-avoidance model, using the best-fit parameters in `parameters.py`:

* **model 1 / `sim1`**: omniscient model, `cohen_avoid` (Equation 1)
* **model 2 / `sim2`**: visual model, `cohen_avoid2` (Equation 2)

Outputs: example-trial simulations (Figures 10, 18), mean distance error, passing-order accuracy and BIC
(Tables 2, 5), model passing order by condition (Figures 7b–c, 15b–c), model SMPD (Figure 12, Tables 3, 6),
and the leave-one-subject-out cross-validation errors (Tables A1, A2) written to `results/` for `stats/stats.R`.

The cross-validation cells read the fitting logs produced by `run_fitting.bat` from `results/fitting/`.
Run `01_import_data.ipynb` first.

In [ ]:
import os
import sys
import json
from collections import Counter
from scipy.integrate import solve_ivp
from scipy.stats import ttest_rel
import numpy as np
from numpy.linalg import norm
from numpy import sqrt
from math import pi
import pickle
import matplotlib.pyplot as plt
from matplotlib import rcParams
from packages import data_container
from packages.data_container import Data
from packages.helper import play_trajs, rotate, sp2a, v2sp, dist, psi, beta, d_theta, d_psi, sp2v, dist, \
    traj_speed, min_sep
from packages.ode_simulator import ODESimulator
from packages.models import cohen_avoid_heading
from sklearn.metrics import mean_squared_error
from fitting import Cohen_movObst1, Cohen_movObst2
from parameters import approaches, avoidances
from config import processed_path, FITTING_DIR, RESULTS_DIR

# For pickle to load the Data object, which is defined in packages.data_container
sys.modules['data_container'] = data_container

data = None
def load_data(filename):
    global data
    with open(processed_path(filename), 'rb') as f:
        data = pickle.load(f)

os.makedirs(RESULTS_DIR, exist_ok=True)

## Experiment 1 (`Cohen_movObst1`)

In [ ]:
'''Simulate one trial avoidance'''
load_data('Cohen_movObst1_data')
############
i = 20
approach = approaches['Fajen_steer1a']['fajen_approach']['differential_evolution'][-1]
avoid = avoidances['Cohen_movObst1']['cohen_avoid2']['differential_evolution']['-1obst_onset']
# avoid = avoidances['Cohen_movObst1']['cohen_avoid_heading']['differential_evolution'][-1]
model_name = 'Visual Model'
############
%matplotlib qt
models = [approach, avoid]
sim = ODESimulator(data=data, ref=[0, 1])
sim.models = models
sim.simulate_all(trials=[i], t_start='obst_onset', t_end='obst_out', ps='trial')
dist_error = sim.test('p_dist')
heading_error = sim.test('phi_RMSE') * 180 / np.pi
speed_error = sim.test('s_RMSE')
t_length = (sim.data.info['obst_out'][i] - sim.data.info['obst_onset'][i]) / sim.data.Hz
title =  model_name + \
        ', subj ' + str(sim.data.info['subj_id'][i]) + \
        ', trial ' + str(sim.data.info['trial_id'][i]) + \
        ', obst path angle: ' + str(sim.data.info['obst_angle'][i]) + '°' + \
        ', obst speed: ' + str(sim.data.info['obst_speed'][i]) + ' m/s' + \
        ',\n distance error: ' + str(round(dist_error, 2)) + ' m' + \
        ', heading RMSE: ' + str(round(heading_error, 2)) + '°' + \
        ', speed RMSE: ' + str(round(speed_error, 2)) + ' m/s'
print(f'Simulation time {t_length} s, distance error {dist_error} m')
sim.play(linestyles=['--', '-', '-', '-'], plot=True, fontsize=13, xrange=[-1, 1], yrange=[1, 5],
         title=title, fig_size=[10, 3.5], save=False)

# plt.gcf().get_axes()[0].legend(loc='lower right')

In [ ]:
'''Simulate all trials, model 1 = omniscient model (cohen_avoid)'''
load_data('Cohen_movObst1_data')
############
# subject = 1
approach = approaches['Fajen_steer1a']['fajen_approach']['differential_evolution'][-1]
avoid = avoidances['Cohen_movObst1']['cohen_avoid']['differential_evolution']['-1obst_onset']

############
%matplotlib qt
models = [approach, avoid]
subjects = range(20)
sim1, trials1 = Cohen_movObst1(subjects)
sim1.reset()
sim1.models = models

# for i in range(len(data.trajs)):
#     if i in data.dump or data.info['subj_id'][i] != subject or data.info['obst_speed'][i] == 0:
#         continue
#     trials.append(i)
sim1.simulate_all(trials1, t_start='obst_onset', t_end='obst_out', ps='trial')
errors1 = sim1.test('p_dist', all_errors=True)
print(f"distance error is {np.mean(errors1)} sd is {np.std(errors1)}")
print(f"order accuracy is {sim1.test('order_accuracy')}")
print(f"heading RMSE is {sim1.test('phi_RMSE')}")
print(f"speed RMSE is {sim1.test('s_RMSE')}")
errors1 = sim1.test('p_dist', all_errors=True)
# Compute BIC
'''BIC = n*ln(MSE) + k*ln(n))'''
k = len(models[1]) - 2
mses = [x**2 for x in errors1]
# n = len(mses)
# n is the n_subj * n_conditions
n_angle = len([x for x in set(data.info['obst_angle']) if x and abs(x) != 0])
n_speed = len([x for x in set(data.info['obst_speed']) if x and abs(x) != 0])
n = len(set(data.info['subj_id'])) * n_angle * n_speed
BIC = n * np.log(np.mean(mses)) + k * np.log(n)
print(f'The BIC of model {sim1.models[1]["name"]} is {BIC}, MSE is {np.mean(mses)}, k is {k}, n is {n}')

In [ ]:
'''Simulate all trials, model 2 = visual model (cohen_avoid2)'''
load_data('Cohen_movObst1_data')
############
# subject = 1
approach = approaches['Fajen_steer1a']['fajen_approach']['differential_evolution'][-1]
avoid = avoidances['Cohen_movObst1']['cohen_avoid2']['differential_evolution']['-1obst_onset']

############
%matplotlib qt
models = [approach, avoid]
subjects = range(20)
sim2, trials2 = Cohen_movObst1(subjects)
sim2.reset()
sim2.models = models

# for i in range(len(data.trajs)):
#     if i in data.dump or data.info['subj_id'][i] != subject or data.info['obst_speed'][i] == 0:
#         continue
#     trials.append(i)
sim2.simulate_all(trials2, t_start='obst_onset', t_end='obst_out', ps='trial')
errors2 = sim2.test('p_dist', all_errors=True)
print(f"distance error is {np.mean(errors2)} sd is {np.std(errors2)}")
print(f"order accuracy is {sim2.test('order_accuracy')}")
print(f"heading RMSE is {sim2.test('phi_RMSE')}")
print(f"speed RMSE is {sim2.test('s_RMSE')}")
errors2 = sim2.test('p_dist', all_errors=True)
# Compute BIC
'''BIC = n*ln(MSE) + k*ln(n))'''
k = len(models[1]) - 2
mses = [x**2 for x in errors2]
# n is the n_subj * n_conditions (same as model 1)
n_angle = len([x for x in set(data.info['obst_angle']) if x and abs(x) != 0])
n_speed = len([x for x in set(data.info['obst_speed']) if x and abs(x) != 0])
n = len(set(data.info['subj_id'])) * n_angle * n_speed
BIC = n * np.log(np.mean(mses)) + k * np.log(n)
print(f'The BIC of model {sim2.models[1]["name"]} is {BIC}, MSE is {np.mean(mses)}, k is {k}, n is {n}')

In [ ]:
'''Cross validation error'''
load_data('Cohen_movObst1_data')
def cross_validate(folder):
    path = os.path.join(FITTING_DIR, folder)
    filenames = [os.path.join(path, name) for name in os.listdir(path) if name[-3:] == 'txt']
    errors = {}
    for filename in filenames:
        with open(filename, 'rb') as f:
            best = 0
            e_min = float('inf')
            if filename[-6] == '_' or filename[-6] == '-':
                subj_id = int(filename[-5])
            elif filename[-6] == 'l':
                subj_id = -1
            else:
                subj_id = int(filename[-6:-4])
            for i, line in enumerate(f):
                if i > 10:
                    try:
                        err = str(line).split("\\t")[2][:10]
                        if err[0] == '0':
                            err = float(err)
                            if err < e_min:
                                e_min = err
                                best = str(line).replace("\\", "")
                    except:
                        pass
        try:
            if 'acceleration' in path:
                models = json.loads(''.join(best.split('t')[1] + best.split('t')[2]).replace("\'", "\"")) # If contains acceleration model
                models[0]['name'] = "acceleration_approach" # If contains acceleration model
            else:
                models = json.loads(best.split('t')[1].replace("\'", "\"")) # If not contains acceleration model
            subjects = [int(subj_id)]
            sim, trials = Cohen_movObst1(subjects)
            sim.reset()
            sim.models = models
            sim.simulate_all(trials, t_start='obst_onset', t_end='obst_out', ps='trial')
            errors[subj_id] = sim.test('p_dist', all_errors=True)
            print(f'{subj_id} finished')
            print(f'filename {filename}')
            print(f'models {sim.models}')
            print(f'error {np.mean(errors[subj_id])}')
        except Exception as e:
            print(e)
    return errors

In [ ]:
'''Output errors for analysis in R'''
out_csv = os.path.join(RESULTS_DIR, 'Cohen_movObst1_CV_errors_obst_onset.csv')
with open(out_csv, 'w') as f:
    f.write('model,subject,error\n')
for i in ['', '2']:
    model = 'cohen_avoid' + i
    errors = cross_validate('Cohen_movObst1_cross_validation_cohen_avoid' + i + '_obst_onset')
    with open(out_csv, 'a') as f:
        for subj, error in errors.items():
            for err in error:
                log = [model, subj, err]
                log = [str(x) for x in log]
                f.write(','.join(log) + '\n')
            

In [ ]:
'''Minimum Passing Distance'''
%matplotlib qt
load_data('Cohen_movObst1_data')
sim = sim1  # sim1 = omniscient model, sim2 = visual model
fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.5), constrained_layout=True)
ax.set_title('Avoidance Model 1')
ax.set_ylabel('SMPD (m)')
ax.set_xlabel('normalized time (%)')
ax.set_ylim((-3, 3))
last_smpd_front = []
last_smpd_behind = []
for j, i in enumerate(sim.i_trials):
    t0 = data.info['obst_onset'][i]
    t1 = data.info['obst_out'][i]
    p0 = sim.p_pred[j]
    p1 = data.info['p_obst'][i][t0:t1]
    v0 = sim.v_pred[j]
    v1 = data.info['v_obst'][i][t0:t1]
    t = np.linspace(0, 100, len(p0))
    smpd = []
    for _p0, _p1, _v0, _v1 in zip(p0, p1, v0, v1):        
        smpd.append(min_sep(_p0, _p1, _v0, _v1)[0])
    ax.plot(t, smpd, 'k', linewidth=0.1, alpha=0.5)
    if smpd[-1] > 0:
        last_smpd_front.append(smpd[-1])
    else:
        last_smpd_behind.append(smpd[-1])
print(np.mean(last_smpd_front), np.std(last_smpd_front), np.mean(last_smpd_behind), np.std(last_smpd_behind))

In [ ]:
'''Minimum Passing Distance by condition'''
load_data('Cohen_movObst1_data')
sim = sim1  # sim1 = omniscient model, sim2 = visual model
angles = sorted(set([abs(x) for x in data.info['obst_angle'] if x]))
speeds = sorted(set([abs(x) for x in data.info['obst_speed'] if x]))
fig, axs = plt.subplots(3, 3, figsize=(12, 8), constrained_layout=True)
fig.suptitle('Signed minimum predicted distance (SMPD) of Omniscient Model')
for ix, angle in enumerate(angles):
    for iy, speed in enumerate(speeds):
        axs[ix, iy].set_title(f'angle {angle}°, speed {speed} m/s')
        axs[ix, iy].set_ylabel('SMPD (m)')
        axs[ix, iy].set_xlabel('normalized time (%)')
        axs[ix, iy].set_ylim((-2, 2))
        for j, i in enumerate(sim.i_trials):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            p0 = sim.p_pred[j]
            p1 = data.info['p_obst'][i][t0:t1]
            v0 = sim.v_pred[j]
            v1 = data.info['v_obst'][i][t0:t1]
            t = np.linspace(0, 100, len(p0))
            smpd = []
            for _p0, _p1, _v0, _v1 in zip(p0, p1, v0, v1):        
                smpd.append(min_sep(_p0, _p1, _v0, _v1)[0])
            axs[ix, iy].plot(t, smpd, 'k', linewidth=0.1, alpha=0.7)

In [ ]:
'''Heatmap of pass order ratio by condition'''
%matplotlib qt
sim = sim2  # sim1 = omniscient model, sim2 = visual model
data = sim.data
name = data.info['experiment_name']
conditions = np.zeros((3, 3)) # angle by speed
angle_ins = {70: 0, 90: 1, 110: 2}
speed_ins = {0.4: 0, 0.6: 1, 0.8: 2}
for angle in angle_ins.keys():
    for speed in speed_ins.keys():
        orders = []
        for j in range(len(sim.pass_order_pred)):
            i = sim.i_trials[j]
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            orders.append(sim.pass_order_pred[j])
        conditions[angle_ins[angle], speed_ins[speed]] = round(np.mean(np.array(orders) > 0), 2)

# Make a pandas df with subj_id, trial_id, obst_angle, obst_speed, pass_order
# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar()
cbar.set_label('Ahead passing ratio (Visual Model)')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] >= 0.7:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 3), list(speed_ins.keys()))
plt.yticks(range(0, 3), list(angle_ins.keys()))
plt.xlabel("Obstacle speed (m/s)")
plt.ylabel("Obstacle path angle (°)")

# Line plot
# plt.figure()
# x = list(speed_ins.keys())
# for i, angle in zip(range(conditions.shape[1]), angle_ins.keys()):
#     plt.plot(x, conditions[:, i], label=f'{angle}°', color='r', alpha=(angle-67.5)/90)
# plt.ylabel("Front pass ratio")
# plt.xlabel("speed (m/s)")
# plt.legend()
# plt.title(name)

## Experiment 2 (`Cohen_movObst2`)

In [ ]:
'''Simulate one trial avoidance'''
load_data('Cohen_movObst2_data')
############
i = 40
approach = approaches['Fajen_steer1a']['fajen_approach']['differential_evolution'][-1]
avoid = avoidances['Cohen_movObst2']['cohen_avoid2']['differential_evolution']['-1obst_onset']
# avoid = avoidances['Cohen_movObst2']['cohen_avoid_heading']['differential_evolution'][-1]
model_name = 'Visual Model'
############
%matplotlib qt
models = [approach, avoid]
sim = ODESimulator(data=data, ref=[0, 1])
sim.models = models
sim.simulate_all(trials=[i], t_start='obst_onset', t_end='obst_out', ps='trial')
dist_error = sim.test('p_dist')
heading_error = sim.test('phi_RMSE') * 180 / np.pi
speed_error = sim.test('s_RMSE')
t_length = (sim.data.info['obst_out'][i] - sim.data.info['obst_onset'][i]) / sim.data.Hz
title = model_name + \
        ', subj ' + str(sim.data.info['subj_id'][i]) + \
        ', trial ' + str(sim.data.info['trial_id'][i]) + \
        ', obst path angle: ' + str(sim.data.info['obst_angle'][i]) + '°' + \
        ', obst init dist: ' + str(sim.data.info['obst_dist'][i]) + 'm' + \
        ',\n distance error: ' + str(round(dist_error, 2)) + ' m' + \
        ', heading RMSE: ' + str(round(heading_error, 2)) + '°' + \
        ', speed RMSE: ' + str(round(speed_error, 2)) + ' m/s'
print(f'Simulation time {t_length} s, distance error {dist_error} m')
sim.play(linestyles=['--', '-', '-', '-'], plot=True, fontsize=13, xrange=[-1.5, 1.5], yrange=[1.5, 6.5],
        title=title, fig_size=[10, 3.5], save=False)

In [ ]:
'''Simulate all trials, model 1 = omniscient model (cohen_avoid)'''
load_data('Cohen_movObst2_data')
############
# subject = 1
approach = approaches['Fajen_steer1a']['fajen_approach']['differential_evolution'][-1]
avoid = avoidances['Cohen_movObst2']['cohen_avoid']['differential_evolution']['-1obst_onset']

############
%matplotlib qt
models = [approach, avoid]
subjects = range(20)
sim1, trials1 = Cohen_movObst2(subjects)
sim1.reset()
sim1.models = models

# for i in range(len(data.trajs)):
#     if i in data.dump or data.info['subj_id'][i] != subject or data.info['obst_speed'][i] == 0:
#         continue
#     trials.append(i)
sim1.simulate_all(trials1, t_start='obst_onset', t_end='obst_out', ps='trial')
errors1 = sim1.test('p_dist', all_errors=True)
print(f"distance error is {np.mean(errors1)} sd is {np.std(errors1)}")
print(f"order accuracy is {sim1.test('order_accuracy')}")
print(f"heading RMSE is {sim1.test('phi_RMSE')}")
print(f"speed RMSE is {sim1.test('s_RMSE')}")
errors1 = sim1.test('p_dist', all_errors=True)
# Compute BIC
'''BIC = n*ln(MSE) + k*ln(n))'''
k = len(models[1]) - 2
mses = [x**2 for x in errors1]
# n = len(mses)
# n is the n_subj * n_conditions
n_angle = len([x for x in set(data.info['obst_angle']) if x])
n_dist = len([x for x in set(data.info['obst_dist']) if x])
n = len(set(data.info['subj_id'])) * n_angle * n_dist
BIC = n * np.log(np.mean(mses)) + k * np.log(n)
print(f'The BIC of model {sim1.models[1]["name"]} is {BIC}, MSE is {np.mean(mses)}, k is {k}, n is {n}')

In [ ]:
'''Simulate all trials, model 2 = visual model (cohen_avoid2)'''
load_data('Cohen_movObst2_data')
############
# subject = 1
approach = approaches['Fajen_steer1a']['fajen_approach']['differential_evolution'][-1]
avoid = avoidances['Cohen_movObst2']['cohen_avoid2']['differential_evolution']['-1obst_onset']

############
%matplotlib qt
models = [approach, avoid]
subjects = range(20)
sim2, trials2 = Cohen_movObst2(subjects)
sim2.reset()
sim2.models = models

# for i in range(len(data.trajs)):
#     if i in data.dump or data.info['subj_id'][i] != subject or data.info['obst_speed'][i] == 0:
#         continue
#     trials.append(i)
sim2.simulate_all(trials2, t_start='obst_onset', t_end='obst_out', ps='trial')
errors2 = sim2.test('p_dist', all_errors=True)
print(f"distance error is {np.mean(errors2)} sd is {np.std(errors2)}")
print(f"order accuracy is {sim2.test('order_accuracy')}")
print(f"heading RMSE is {sim2.test('phi_RMSE')}")
print(f"speed RMSE is {sim2.test('s_RMSE')}")
errors2 = sim2.test('p_dist', all_errors=True)
# Compute BIC
'''BIC = n*ln(MSE) + k*ln(n))'''
k = len(models[1]) - 2
mses = [x**2 for x in errors2]
# n = len(mses)
# n is the n_subj * n_conditions
n_angle = len([x for x in set(data.info['obst_angle']) if x])
n_dist = len([x for x in set(data.info['obst_dist']) if x])
n = len(set(data.info['subj_id'])) * n_angle * n_dist
BIC = n * np.log(np.mean(mses)) + k * np.log(n)
print(f'The BIC of model {sim2.models[1]["name"]} is {BIC}, MSE is {np.mean(mses)}, k is {k}, n is {n}')

In [ ]:
'''Cross validation error'''
load_data('Cohen_movObst2_data')
def cross_validate(folder):
    path = os.path.join(FITTING_DIR, folder)
    filenames = [os.path.join(path, name) for name in os.listdir(path) if name[-3:] == 'txt']
    errors = {}
    for filename in filenames:
        with open(filename, 'rb') as f:
            best = 0
            e_min = float('inf')
            if filename[-6] == '_' or filename[-6] == '-':
                subj_id = int(filename[-5])
            elif filename[-6] == 'l':
                subj_id = -1
            else:
                subj_id = int(filename[-6:-4])
            for i, line in enumerate(f):
                if i > 10:
                    try:
                        err = str(line).split("\\t")[2][:10]
                        if err[0] == '0':
                            err = float(err)
                            if err < e_min:
                                e_min = err
                                best = str(line).replace("\\", "")
                    except:
                        pass
        try:
            if 'acceleration' in path:
                models = json.loads(''.join(best.split('t')[1] + best.split('t')[2]).replace("\'", "\"")) # If contains acceleration model
                models[0]['name'] = "acceleration_approach" # If contains acceleration model
            else:
                models = json.loads(best.split('t')[1].replace("\'", "\"")) # If not contains acceleration model
            subjects = [int(subj_id)]
            sim, trials = Cohen_movObst2(subjects)
            sim.reset()
            sim.models = models
            sim.simulate_all(trials, t_start='obst_onset', t_end='obst_out', ps='trial')
            errors[subj_id] = sim.test('p_dist', all_errors=True)
            print(f'{subj_id} finished')
            print(f'filename {filename}')
            print(f'models {sim.models}')
            print(f'error {np.mean(errors[subj_id])}')
        except Exception as e:
            print(e)
    return errors

In [ ]:
'''Output errors for analysis in R'''
out_csv = os.path.join(RESULTS_DIR, 'Cohen_movObst2_CV_errors_obst_onset.csv')
with open(out_csv, 'w') as f:
    f.write('model,subject,error\n')
for i in ['', '2']:
    model = 'cohen_avoid' + i
    errors = cross_validate('Cohen_movObst2_cross_validation_cohen_avoid' + i + '_obst_onset')
    with open(out_csv, 'a') as f:
        for subj, error in errors.items():
            for err in error:
                log = [model, subj, err]
                log = [str(x) for x in log]
                f.write(','.join(log) + '\n')
            

In [ ]:
'''Minimum Passing Distance'''
%matplotlib qt
load_data('Cohen_movObst2_data')
sim = sim2  # sim1 = omniscient model, sim2 = visual model
fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.5), constrained_layout=True)
ax.set_title('Avoidance Model 1')
ax.set_ylabel('SMPD (m)')
ax.set_xlabel('normalized time (%)')
ax.set_ylim((-3, 3))
last_smpd_front = []
last_smpd_behind = []
for j, i in enumerate(sim.i_trials):
    t0 = data.info['obst_onset'][i]
    t1 = data.info['obst_out'][i]
    p0 = sim.p_pred[j]
    p1 = data.info['p_obst'][i][t0:t1]
    v0 = sim.v_pred[j]
    v1 = data.info['v_obst'][i][t0:t1]
    t = np.linspace(0, 100, len(p0))
    smpd = []
    for _p0, _p1, _v0, _v1 in zip(p0, p1, v0, v1):        
        smpd.append(min_sep(_p0, _p1, _v0, _v1)[0])
    ax.plot(t, smpd, 'k', linewidth=0.1, alpha=0.5)
    if smpd[-1] > 0:
        last_smpd_front.append(smpd[-1])
    else:
        last_smpd_behind.append(smpd[-1])
print(np.mean(last_smpd_front), np.std(last_smpd_front), np.mean(last_smpd_behind), np.std(last_smpd_behind))

In [ ]:
'''Minimum Passing Distance by condition'''
load_data('Cohen_movObst2_data')
sim = sim2  # sim1 = omniscient model, sim2 = visual model
angles = sorted(set([abs(x) for x in data.info['obst_angle'] if x]))
dists = sorted(set([abs(x) for x in data.info['obst_dist'] if x]))
fig, axs = plt.subplots(5, 2, figsize=(8, 10), constrained_layout=True)
fig.suptitle('Signed minimum predicted distance (SMPD) of Visual Model')
for ix, angle in enumerate(angles):
    for iy, dist in enumerate(dists):
        axs[ix, iy].set_title(f'angle {angle}°, distance {dist} m')
        axs[ix, iy].set_ylabel('SMPD (m)')
        axs[ix, iy].set_xlabel('normalized time (%)')
        axs[ix, iy].set_ylim((-3, 3))
        for j, i in enumerate(sim.i_trials):
            if not data.info['obst_angle'][i]:
                continue
            if angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            p0 = sim.p_pred[j]
            p1 = data.info['p_obst'][i][t0:t1]
            v0 = sim.v_pred[j]
            v1 = data.info['v_obst'][i][t0:t1]
            t = np.linspace(0, 100, len(p0))
            smpd = []
            for _p0, _p1, _v0, _v1 in zip(p0, p1, v0, v1):        
                smpd.append(min_sep(_p0, _p1, _v0, _v1)[0])
            axs[ix, iy].plot(t, smpd, 'k', linewidth=0.1, alpha=0.7)

In [ ]:
'''Heatmap and line plot of front passing ratio'''
%matplotlib qt
sim = sim2  # sim1 = omniscient model, sim2 = visual model
data = sim.data
name = data.info['experiment_name']
conditions = np.zeros((2, 5)) # dist by angle
angle_ins = {100: 0, 110: 1, 120: 2, 130: 3, 140: 4}
dist_ins = {4.7: 0, 5.8: 1}
for angle in angle_ins.keys():
    for dist in dist_ins.keys():
        orders = []
        for j in range(len(sim.pass_order_pred)):
            i = sim.i_trials[j]
            if not data.info['obst_angle'][i]:
                continue
            if angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            orders.append(sim.pass_order_pred[j])
        conditions[dist_ins[dist], angle_ins[angle]] = round(np.mean(np.array(orders) > 0), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar(orientation="horizontal")
cbar.set_label('Ahead passing ratio (Visual Model)')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.7:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 5), list(angle_ins.keys()))
plt.yticks(range(0, 2), list(dist_ins.keys()))
plt.xlabel("Obstacle path angle (°)")
plt.ylabel("Init obstacle dist (m)")